# Proteome exploration with ESMC embeddings

A **single** scanpy graph drives everything: a KNN graph on the ESMC embeddings, one UMAP layout, and Leiden clusters — same neighbors graph, fixed seed. SAE features are tested for enrichment per cluster (proteins as "cells", SAE features as "genes") and richly annotated from the ESM Atlas.

**Prerequisites:** run the embedding step first, and `pip install -e "..[cluster]"` (scanpy, leidenalg, igraph). Needs `BASEROW_TOKEN` / `BIOHUB_API_TOKEN` in the env.

In [ ]:
from och_annotate.config import load_config
from och_annotate.analysis import load_embeddings

cfg = load_config("../config/octopus_chierchiae.yaml")
df = load_embeddings(cfg, prefer_cache=True)   # backfills Baserow metadata cols not in cache
df = df.rename(columns={"orthogroup": "Orthogroup"})   # nicer hover label
print(f"Loaded {len(df)} embeddings; vector dim = {len(df['embedding'].iloc[0])}")
df.head()

## One graph: KNN → UMAP → Leiden

Defaults match the original UMAP (`n_neighbors=15`, `min_dist=0.1`, cosine). Tune **granularity**: `N_NEIGHBORS`/`MIN_DIST` shape the UMAP, `LEIDEN_RES` the cluster count.

In [ ]:
import scanpy as sc
from och_annotate.analysis import build_anndata, sae_enrichment, plot_umap

SEED        = 0
N_NEIGHBORS = 15
MIN_DIST    = 0.1
METRIC      = "cosine"
LEIDEN_RES  = 1.0       # clustering granularity (higher = more clusters)

adata = build_anndata(df)
sc.pp.neighbors(adata, use_rep="X_esmc", n_neighbors=N_NEIGHBORS, metric=METRIC, random_state=SEED)
sc.tl.umap(adata, min_dist=MIN_DIST, random_state=SEED)
sc.tl.leiden(adata, resolution=LEIDEN_RES, flavor="igraph", n_iterations=2,
             directed=False, random_state=SEED)

meta_cols = [c for c in df.columns if c not in ("embedding", "sae_top_features")]
coords = df[meta_cols].copy().reset_index(drop=True)
coords["umap_0"] = adata.obsm["X_umap"][:, 0]
coords["umap_1"] = adata.obsm["X_umap"][:, 1]
coords["leiden"] = adata.obs["leiden"].to_numpy()
print(f"{adata.n_obs} proteins; {coords['leiden'].nunique()} Leiden clusters")

In [ ]:
# UMAP colored by chromosome; hover shows gene + mouse ortholog + orthogroup
plot_umap(coords, color="chromosome", hover=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "Orthogroup", "chromosome", "leiden"],
          title=f"{cfg.name} — UMAP (chromosome)").show()

In [ ]:
# Same UMAP colored by Leiden cluster; same hover fields
plot_umap(coords, color="leiden", hover=["transcript_id", "gene_id", "Ochierchiae_name", "Mmusculus_gene_name", "Orthogroup", "chromosome", "leiden"],
          title=f"{cfg.name} — UMAP (Leiden clusters)").show()

## SAE-feature enrichment per cluster

Wilcoxon rank-sum on the SAE activation matrix, annotated from the **ESM Atlas**: `label`, `category`, `activation_pattern`, `exemplar_protein_families`, top **SwissProt** proteins, and `uniref90_idf` — used to **IDF-weight** markers toward specific (rare) features.

In [ ]:
import pandas as pd
from och_annotate.atlas import fetch_feature_descriptions

enrich = sae_enrichment(adata, groupby="leiden", method="wilcoxon", n=15)

# Rich per-feature Atlas metadata for the enriched features (concurrent, cached; no credits)
feat_ids = sorted(enrich["sae_feature"].astype(int).unique())
meta = fetch_feature_descriptions(feat_ids, cache_path="../data/sae_feature_metadata.parquet")
keep = ["feature", "label", "category", "activation_pattern",
        "exemplar_protein_families", "uniref90_idf", "swissprot_top"]
meta = meta[keep].copy(); meta["feature"] = meta["feature"].astype(str)

enrich["feature"] = enrich["sae_feature"].astype(str)
enrich = enrich.merge(meta, on="feature", how="left").drop(columns="feature")

# IDF-weighting: upweight features that are rare across UniRef90 (more specific).
enrich["idf"] = pd.to_numeric(enrich["uniref90_idf"], errors="coerce").fillna(1.0)
enrich["score_idf"] = enrich["scores"] * enrich["idf"]

enrich.to_csv("../data/cluster_sae_enrichment.csv", index=False)
print(f"Annotated {len(feat_ids)} features (category / IDF / exemplars / SwissProt); "
      f"{len(enrich)} rows across {enrich['leiden'].nunique()} clusters")

In [ ]:
# Per-cluster functional profile: the category mix of each cluster's top-15 features
profile = (enrich.assign(cluster=enrich["leiden"].astype(int))
                 .groupby("cluster")["category"]
                 .apply(lambda s: ", ".join(f"{c} ({n})" for c, n in
                        s.fillna("(uncat)").replace("", "(uncat)").value_counts().head(4).items()))
                 .rename("top_feature_categories").to_frame())
with pd.option_context("display.max_rows", None, "display.max_colwidth", 90):
    display(profile)

In [ ]:
from IPython.display import display

# Top-5 per cluster, ranked by the IDF-WEIGHTED score (specific features rise).
top5 = (enrich.assign(cluster=enrich["leiden"].astype(int))
              .sort_values(["cluster", "score_idf"], ascending=[True, False])
              .groupby("cluster", observed=True).head(5).copy())
top5["rank"] = top5.groupby("cluster").cumcount() + 1
view = top5[["cluster", "rank", "sae_feature", "label", "category", "scores", "idf", "score_idf"]]

def _shade(row):
    tint = "#eef3fa" if row.name[0] % 2 == 0 else "#ffffff"
    return [f"background-color: {tint}"] * len(row)

styled = (view.set_index(["cluster", "rank"]).style
              .format({"scores": "{:.1f}", "idf": "{:.2f}", "score_idf": "{:.1f}"})
              .apply(_shade, axis=1)
              .set_properties(**{"text-align": "left"})
              .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    display(styled)

In [ ]:
# Rich context for each cluster's lead (IDF-weighted top) feature
lead = top5[top5["rank"] == 1].sort_values("cluster")
for r in lead.itertuples():
    ap = (str(r.activation_pattern) or "").strip().replace("\n", " ")
    ex = (str(r.exemplar_protein_families) or "").strip().splitlines()
    print(f"\u2501\u2501 cluster {r.cluster}  [{r.sae_feature}] {r.label}  ({r.category})")
    print(f"     activation : {ap[:220]}")
    print(f"     exemplars  : {(ex[0][:200] if ex else '')}")
    print(f"     SwissProt  : {r.swissprot_top}")

In [ ]:
# Dotplot of marker SAE features across clusters (Wilcoxon ranking)
sc.tl.dendrogram(adata, groupby="leiden")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)

## Per-cluster feature profile — salience × ubiquity

A second lens on cluster identity, ranked by **mean normalized activation** (what ESMC finds most *salient* across the cluster) with **occurrence** = the fraction of members in which the feature is active. Universal features (occurrence ≈ 100%) define the family; partial ones (≈ 30–60%) flag subgroups or domain variants. This complements the differential Wilcoxon markers above. Reporting the **top 10** per cluster — ranks 6–15 are where subfamily discrimination lives.

> **Residue regions** (start/end/peak per feature) would be the next upgrade, but our cache max-pools SAE activations to one value per protein — positions were discarded. Recovering them needs a per-residue SAE re-run (Biohub credits).

In [ ]:
from och_annotate.analysis import cluster_feature_profile

# Top-10 features per cluster by mean normalized activation (+ occurrence rate)
profile = cluster_feature_profile(adata, groupby="leiden", n=10)

# Atlas labels/category for the profile features (cached; no Biohub credits)
pf_ids = sorted(profile["sae_feature"].unique())
pmeta = fetch_feature_descriptions(pf_ids, cache_path="../data/sae_feature_metadata.parquet")
plabel = dict(zip(pmeta["feature"].astype(int), pmeta["label"]))
pcat = dict(zip(pmeta["feature"].astype(int), pmeta["category"]))
profile["label"] = profile["sae_feature"].map(plabel)
profile["category"] = profile["sae_feature"].map(pcat)
profile.to_csv("../data/cluster_feature_profile.csv", index=False)
print(f"Profiled {profile['cluster'].nunique()} clusters x top-10 features "
      f"({len(pf_ids)} unique features)")

In [ ]:
# Grouped top-10 profile per cluster: salience (mean_activation) + ubiquity (occurrence)
pv = (profile.assign(cluster=lambda d: d["cluster"].astype(int))
             .sort_values(["cluster", "rank"])
             [["cluster", "rank", "sae_feature", "label", "category",
               "mean_activation", "occurrence"]])

def _shade2(row):
    tint = "#eef3fa" if row.name[0] % 2 == 0 else "#ffffff"
    return [f"background-color: {tint}"] * len(row)

styled = (pv.set_index(["cluster", "rank"]).style
            .format({"mean_activation": "{:.3f}", "occurrence": "{:.0%}"})
            .apply(_shade2, axis=1)
            .set_properties(**{"text-align": "left"})
            .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    display(styled)

### Notes on the Atlas annotations

All metadata comes from the public **ESM Atlas** feature API
(`biohub.ai/esm/protein/api/v1alpha1/features/{idx}`) via
`och_annotate.atlas.fetch_feature_descriptions` — cached under `data/`, **not** charged
against Biohub embedding credits. The grouped table is ranked by
`score_idf = wilcoxon_score × uniref90_idf` so cluster-specific (rare) features rise above
ubiquitous ones; the lead-feature block adds activation pattern, exemplar families and
reviewed-UniProt examples. Full table: `data/cluster_sae_enrichment.csv`.

### Other next steps
- Write `adata.obs["leiden"]` back to Baserow as a `leiden_cluster` column.
- GO-enrich each cluster from the SwissProt example proteins.
- Tune `LEIDEN_RES`, `N_NEIGHBORS`, `MIN_DIST`.